In [0]:
# =============================================================================
# Notebook  : 02b_map_01_contacts_all
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_01_contacts_all
# Spec Ref  : §1.2 Refresh Materialized Views
# Purpose   : Build contacts_all Delta table = contacts UNION ALL crm_contacts
#             (currently just hgi.silver.contacts — ready for multi-source union)
#             This MUST run FIRST before any other map notebook.
#             All downstream map queries join against contacts_all.
#
# Why Delta table instead of plain VIEW?
#   A plain VIEW re-scans hgi.silver.contacts on every downstream JOIN.
#   A Delta table is read once, cached on disk, and skipped by Liquid Clustering.
#   For 100k+ contacts this is the difference between seconds and minutes.
#
# Serverless: works on 2X-Small — safe_spark_conf skips unsupported configs
# Job Compute: all SPARK_CONF_MAP configs apply for full performance
# =============================================================================

In [0]:

# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/FINAL_pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# CELL 2 ── Widget
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()

print(f"=== Map 01: contacts_all ===  customer={customer_id}")
print(f"  Reading from : {sv}.contacts")
print(f"  Writing to   : {sv}.contacts_all")

In [0]:
# CELL 3 ── Create contacts_all as a full-replacement Delta table
# CREATE OR REPLACE rebuilds the entire table atomically.
# Uses CLUSTER BY (tenant, email, domain) so downstream JOINs on email
# and domain skip non-matching files automatically.

# Safe drop in case target exists as a VIEW
safe_drop_view(f"{sv}.contacts_all")

spark.sql(f"""
    CREATE OR REPLACE TABLE {sv}.contacts_all
    USING DELTA
    CLUSTER BY (tenant, email, domain)
    {DELTA_TBLPROPS_MAP}
    AS
    SELECT
        id,
        tenant,
        email,
        domain,
        source_system,
        source_system_object,
        source_key_name,
        source_key_value,
        data_timestamp,
        a_accountid
    FROM {sv}.contacts
    WHERE id IS NOT NULL
""")

In [0]:
# CELL 4 ── Verify and report
n = cnt(f"{sv}.contacts_all")
print(f"  contacts_all: {n:,} rows built")

# Spec DQ gate: no null IDs
null_ids = spark.sql(f"SELECT COUNT(*) FROM {sv}.contacts_all WHERE id IS NULL").collect()[0][0]
if null_ids > 0:
    print(f"  WARNING: {null_ids} null IDs in contacts_all")
else:
    print(f"  DQ OK: no null IDs")

dbutils.notebook.exit("Success")
